# M2 reference dump -- Alpamayo-1.5 Referenzszene als Rohdaten

Kein Modell, keine GPU. Laedt die M0-Referenzszene ueber dieselben Datenzugriffe wie der Upstream-Loader und schreibt ROHDATEN (Weltposen vor der Ego-Frame-Transformation, Rohbilder, echte Frame-Zeitstempel) sowie die fertigen Loader-Tensoren als SOLL. Der lokale Adapter muss aus den Rohdaten exakt die Soll-Tensoren erzeugen.

In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, sys

MODEL_ID = 'nvidia/Alpamayo-1.5-10B'
UPSTREAM_URL = 'https://github.com/NVlabs/alpamayo1.5.git'
UPSTREAM_COMMIT = '24179cfa8b2eeaf775e9e21698b23af0f899522d'
WORK = Path('/kaggle/temp/alpamayo_m0')
CACHE = Path('/kaggle/temp/huggingface')
RESULT = Path('/kaggle/working/m0_results/m0_kaggle_attempt.json')
WORK.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
RESULT.parent.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'hub')
os.environ['UV_LINK_MODE'] = 'copy'

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
secret_source = 'environment' if hf_token else None
secret_errors = {}
if not hf_token:
    from kaggle_secrets import UserSecretsClient
    secret_client = UserSecretsClient()
    for secret_label in (
        'HF_TOKEN',
        'HUGGING_FACE_HUB_TOKEN',
        'HUGGINGFACE_TOKEN',
        'HF_ACCESS_TOKEN',
        'huggingface',
    ):
        try:
            candidate = secret_client.get_secret(secret_label)
        except Exception as exc:
            secret_errors[secret_label] = f'{type(exc).__name__}: {exc}'
            continue
        if candidate:
            hf_token = candidate
            secret_source = f'kaggle:{secret_label}'
            break
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
os.environ['M0_SECRET_SOURCE'] = secret_source or 'none'

print(json.dumps({
    'python_base': platform.python_version(),
    'hf_token_present': bool(hf_token),
    'hf_token_source': secret_source,
    'secret_lookup_errors': secret_errors,
    'working_free_gib': round(shutil.disk_usage('/kaggle/working').free / 1024**3, 1),
    'temp_free_gib': round(shutil.disk_usage('/kaggle/temp').free / 1024**3, 1),
}, indent=2))
if not hf_token:
    print('WARNING: HF_TOKEN is missing. Model files are public, but the PhysicalAI dataset may require accepted access and authentication.')

In [ ]:
def run(command, cwd=None):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(command, cwd=cwd, check=True, env=os.environ.copy())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.9.7'], check=True)
UV = shutil.which('uv')
assert UV, 'uv executable was not installed'
REPO = WORK / 'alpamayo1.5'
if not (REPO / '.git').exists():
    run(['git', 'clone', '--filter=blob:none', UPSTREAM_URL, str(REPO)])
run(['git', 'fetch', 'origin', UPSTREAM_COMMIT, '--depth', '1'], cwd=REPO)
run(['git', 'checkout', '--detach', UPSTREAM_COMMIT], cwd=REPO)
run([UV, 'sync', '--no-install-package', 'flash-attn'], cwd=REPO)
PYTHON312 = REPO / '.venv/bin/python'
assert PYTHON312.exists(), 'Python 3.12 environment was not created'
# v20b: explizit in das Runner-Env installieren. Ohne --python loeste uv ein anderes Env auf,
# und der Runner fiel still auf FP16 zurueck (bitsandbytes_import_error in Version 22).
run([UV, 'pip', 'install', '--python', str(PYTHON312), 'bitsandbytes>=0.45.0'], cwd=REPO)
run([str(PYTHON312), '-c', "import bitsandbytes, torch; print({'bitsandbytes': bitsandbytes.__version__, 'cuda': torch.version.cuda})"])
run([str(PYTHON312), '-c', "import platform, torch, transformers; print({'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__})"])

In [ ]:
dump = "from __future__ import annotations\nimport json, os, sys, time, traceback\nfrom pathlib import Path\nimport numpy as np\n\nREPO = Path('/kaggle/temp/alpamayo_m0/alpamayo1.5')\nCACHE = Path('/kaggle/temp/huggingface')\nOUT = Path('/kaggle/working/m2_reference'); OUT.mkdir(parents=True, exist_ok=True)\nsys.path.insert(0, str(REPO / 'src'))\nos.environ['HF_HOME'] = str(CACHE); os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'hub')\n\nCLIP_ID = '030c760c-ae38-49aa-9ad8-f5650a545d26'   # identisch zu M0 (golden/m0_t4x2_nf4_experimental.json)\nT0_US = 5_100_000\nN_HIST, N_FUT, DT = 16, 64, 0.1\nmeta = {'clip_id': CLIP_ID, 't0_us': T0_US, 'status': 'started'}\n\n\ndef write_meta():\n    (OUT / 'meta.json').write_text(json.dumps(meta, indent=2, default=str), encoding='utf-8')\n\n\ndef jsonable(val):\n    try:\n        return json.loads(json.dumps(val, default=lambda o: getattr(o, 'tolist', lambda: str(o))()))\n    except Exception as exc:\n        return f'unserialisierbar: {type(exc).__name__}'\n\n\nwrite_meta()\ntry:\n    import physical_ai_av\n    from alpamayo1_5.load_physical_aiavdataset import load_physical_aiavdataset\n\n    avdi = physical_ai_av.PhysicalAIAVDatasetInterface()\n\n    # --- SOLL: der unveraenderte Upstream-Loader -------------------------------------------\n    t = time.perf_counter()\n    ref = load_physical_aiavdataset(CLIP_ID, t0_us=T0_US, avdi=avdi)\n    meta['loader_seconds'] = time.perf_counter() - t\n    np.savez_compressed(\n        OUT / 'loader_reference.npz',\n        image_frames=ref['image_frames'].numpy(),        # (N_cam, 4, 3, H, W) uint8, nach Index sortiert\n        camera_indices=ref['camera_indices'].numpy(),\n        ego_history_xyz=ref['ego_history_xyz'].numpy(),  # (1,1,16,3) float32, Ego-Frame bei t0\n        ego_history_rot=ref['ego_history_rot'].numpy(),  # (1,1,16,3,3)\n        ego_future_xyz=ref['ego_future_xyz'].numpy(),\n        ego_future_rot=ref['ego_future_rot'].numpy(),\n        relative_timestamps=ref['relative_timestamps'].numpy(),\n        absolute_timestamps=ref['absolute_timestamps'].numpy(),\n    )\n    meta['loader_shapes'] = {k: list(ref[k].shape) for k in (\n        'image_frames', 'camera_indices', 'ego_history_xyz', 'ego_history_rot', 'ego_future_xyz', 'absolute_timestamps')}\n\n    # --- ROH: Weltposen VOR der Transformation, exakt die Datenzugriffe des Loaders ----------\n    egomotion = avdi.get_clip_feature(CLIP_ID, avdi.features.LABELS.EGOMOTION, maybe_stream=True)  # wie der Loader\n    hist_ts = (T0_US + np.arange(-(N_HIST - 1) * DT * 1e6, DT * 1e6 / 2, DT * 1e6)).astype(np.int64)\n    fut_ts = (T0_US + np.arange(DT * 1e6, (N_FUT + 0.5) * DT * 1e6, DT * 1e6)).astype(np.int64)\n    eh = egomotion(hist_ts)\n    ef = egomotion(fut_ts)\n    raw = {\n        'history_timestamps_us': hist_ts,\n        'future_timestamps_us': fut_ts,\n        'world_xyz_history': np.asarray(eh.pose.translation, dtype=np.float64),                 # (16,3)\n        'world_quat_xyzw_history': np.asarray(eh.pose.rotation.as_quat(), dtype=np.float64),    # (16,4) scipy xyzw\n        'world_xyz_future': np.asarray(ef.pose.translation, dtype=np.float64),\n        'world_quat_xyzw_future': np.asarray(ef.pose.rotation.as_quat(), dtype=np.float64),\n    }\n    meta['egomotion_object_attrs'] = sorted(a for a in dir(eh) if not a.startswith('_'))[:40]\n    meta['pose_object_attrs'] = sorted(a for a in dir(eh.pose) if not a.startswith('_'))[:40]\n\n    cams = {\n        'cross_left': avdi.features.CAMERA.CAMERA_CROSS_LEFT_120FOV,\n        'front_wide': avdi.features.CAMERA.CAMERA_FRONT_WIDE_120FOV,\n        'cross_right': avdi.features.CAMERA.CAMERA_CROSS_RIGHT_120FOV,\n        'front_tele': avdi.features.CAMERA.CAMERA_FRONT_TELE_30FOV,\n    }\n    img_ts_req = np.array([T0_US - (4 - 1 - i) * int(DT * 1e6) for i in range(4)], dtype=np.int64)\n    raw['image_timestamps_requested_us'] = img_ts_req\n    cam_meta = {}\n    cam = None\n    for name, feat in cams.items():\n        cam = avdi.get_clip_feature(CLIP_ID, feat, maybe_stream=True)  # wie der Loader\n        frames, frame_ts = cam.decode_images_from_timestamps(img_ts_req)   # (4,H,W,3) uint8, ECHTE ts\n        raw[f'frames_hwc_{name}'] = np.asarray(frames, dtype=np.uint8)\n        raw[f'frame_timestamps_us_{name}'] = np.asarray(frame_ts, dtype=np.int64)\n        cam_meta[name] = {\n            'feature': str(feat), 'shape': list(np.asarray(frames).shape),\n            'requested_us': img_ts_req.tolist(),\n            'actual_us': np.asarray(frame_ts).astype(np.int64).tolist(),\n        }\n        # Kalibrierung, falls am Kameraobjekt abrufbar (Vorarbeit fuer M1) -- best effort, nie ein Fehler\n        for attr in ('intrinsics', 'calibration', 'extrinsics', 'sensor_to_rig', 'camera_model', 'metadata', 'info'):\n            val = getattr(cam, attr, None)\n            if val is not None and not callable(val):\n                cam_meta[name][attr] = jsonable(val)\n    np.savez_compressed(OUT / 'raw_reference.npz', **raw)\n    meta['cameras'] = cam_meta\n    meta['camera_object_attrs'] = sorted(a for a in dir(cam) if not a.startswith('_'))[:60]\n    # Welche Kalibrierungs-Features kennt der Datensatz? (M1)\n    try:\n        feats = avdi.features\n        meta['feature_groups'] = sorted(a for a in dir(feats) if not a.startswith('_'))\n        for grp in ('CALIBRATION', 'SENSOR', 'RIG', 'LABELS'):\n            g = getattr(feats, grp, None)\n            if g is not None:\n                meta[f'features_{grp}'] = sorted(a for a in dir(g) if not a.startswith('_'))[:40]\n    except Exception as exc:\n        meta['feature_probe_error'] = f'{type(exc).__name__}: {exc}'\n    # --- KALIBRIERUNG (M1): rohe Parquet-Zeilen + geparste Objekte ------------------------------\n    import pandas as pd\n    calib = {}\n    chunk = avdi.get_clip_chunk(CLIP_ID)\n    for fname in ('CAMERA_INTRINSICS', 'CAMERA_INTRINSICS_OFFLINE', 'SENSOR_EXTRINSICS',\n                  'SENSOR_EXTRINSICS_OFFLINE', 'VEHICLE_DIMENSIONS'):\n        feat = getattr(avdi.features.CALIBRATION, fname, None)\n        if feat is None:\n            continue\n        entry = {'feature': str(feat)}\n        try:\n            fn = avdi.features.get_chunk_feature_filename(chunk, feat)\n            with avdi.open_file(fn, maybe_stream=True) as f:\n                df = pd.read_parquet(f)\n            entry['parquet_columns'] = [str(c) for c in df.columns]\n            entry['index_names'] = [str(n) for n in df.index.names]\n            rows = df.loc[CLIP_ID]\n            rows = rows.to_frame().T if isinstance(rows, pd.Series) else rows\n            rows.reset_index().to_csv(OUT / f'calibration_{fname.lower()}.csv', index=False)\n            entry['n_rows'] = int(len(rows))\n            entry['row_index'] = [str(i) for i in rows.index]\n        except Exception as exc:\n            entry['raw_error'] = f'{type(exc).__name__}: {exc}'\n        try:\n            obj = avdi.get_clip_feature(CLIP_ID, feat, maybe_stream=True)\n            entry['parsed_type'] = type(obj).__name__\n            if hasattr(obj, 'sensor_poses'):\n                entry['sensor_to_rig_4x4'] = {k: np.asarray(v.as_matrix()).tolist() for k, v in obj.sensor_poses.items()}\n            elif hasattr(obj, 'camera_models'):\n                entry['camera_models'] = {\n                    k: {'type': type(m).__name__, 'width': m.width, 'height': m.height,\n                        'principal_point': np.asarray(getattr(m, 'principal_point', [])).tolist(),\n                        'th2r_coef': np.asarray(getattr(getattr(m, 'th2r', None), 'coef', [])).tolist(),\n                        'r2th_coef': np.asarray(getattr(getattr(m, 'r2th', None), 'coef', [])).tolist()}\n                    for k, m in obj.camera_models.items()}\n            elif hasattr(obj, 'wheelbase'):\n                entry['vehicle'] = {k: float(v) for k, v in vars(obj).items()}\n        except Exception as exc:\n            entry['parse_error'] = f'{type(exc).__name__}: {exc}'\n        calib[fname] = entry\n    meta['calibration'] = calib\n    meta['status'] = 'ok'\nexcept Exception as exc:\n    meta['status'] = 'failed'\n    meta['exception'] = {'type': type(exc).__name__, 'message': str(exc), 'traceback': traceback.format_exc()}\nfinally:\n    write_meta()\n    print(json.dumps({k: v for k, v in meta.items() if k not in ('cameras',)}, indent=2, default=str))\n    for p in sorted(OUT.glob('*')):\n        print(f'{p.stat().st_size / 2**20:8.1f} MiB  {p.name}')\nif meta['status'] != 'ok':\n    raise SystemExit(1)\n"
from pathlib import Path
p = Path('/kaggle/working/m2_dump_runner.py'); p.write_text(dump, encoding='utf-8'); print('wrote', p)

In [ ]:
import subprocess, sys, os
r = subprocess.run([str(PYTHON312), '/kaggle/working/m2_dump_runner.py'], env=os.environ.copy(), check=False)
print('exit', r.returncode)
if r.returncode != 0:
    raise SystemExit(r.returncode)